# DistilBERT fine-tuning for prompt injection detection
I need to use same data and the same evaluation as the TF-IDF + logistic regression notebook, so the two can be compared directly after.

**What this notebook speciffically does**
1. Loads `deepset/prompt-injections` and keeps the **English subset** (German examples are removed cause we wont use the german in the dataset).
2. Checks how long the prompts are in tokens.
3. Evaluates DistilBERT with repeated stratified 5-fold cross-validation on the `train` split. The folds use the same `random_state=42` and `n_splits=5` as the TF-IDF cross-validation, so the comparison is paired.
4. Trains once on all of `train` and evaluates once on the official `test` split. Nothing is tuned on `test`.
5. Measures latency per prompt and saves score files in the shared format `id, label, score, latency_ms`.

We need to start testing the whole notebook first with `N_REPEATS = 1`.

In [5]:
%pip install -q transformers datasets scikit-learn pandas langdetect

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install torch transformers scikit-learn langdetect pandas

   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/126.7 MB 6.8 MB/s eta 0:00:19
    --------------------------------------- 2.4/126.7 MB 6.7 MB/s eta 0:00:19
   - -------------------------------------- 3.7/126.7 MB 6.7 MB/s eta 0:00:19
   - -------------------------------------- 5.0/126.7 MB 6.6 MB/s eta 0:00:19
   -- ------------------------------------- 6.6/126.7 MB 6.7 MB/s eta 0:00:19
   -- ------------------------------------- 7.9/126.7 MB 6.6 MB/s eta 0:00:19
   -- ------------------------------------- 9.2/126.7 MB 6.6 MB/s eta 0:00:18
   --- ------------------------------------ 10.5/126.7 MB 6.6 MB/s eta 0:00:18
   --- ------------------------------------ 11.8/126.7 MB 6.6 MB/s eta 0:00:18
   ---- ----------------------------------- 13.1/126.7 MB 6.6 MB/s eta 0:00:18
   ---- ----------------------------------- 14.4/126.7 MB 6.6 MB/s eta 0:00:18
   ----- ---------------------------------- 16.0/126.7 MB 6.6 MB/s 

In [22]:
import random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from langdetect import DetectorFactory, detect_langs
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import RepeatedStratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

# ---- configuration (write these values down in the methodology) ----
MODEL_NAME = "distilbert-base-uncased"   # English only (use "distilbert-base-multilingual-cased" for other languages)
MAX_LEN    = 128      # prompts longer than this (in tokens) are truncated
BATCH_SIZE = 16
EPOCHS     = 3
LR         = 2e-5
SEED       = 42
N_SPLITS, N_REPEATS = 5, 2

METHOD_NAME = "distilbert_multilingual" if "multilingual" in MODEL_NAME else "distilbert"

if torch.cuda.is_available():
    device, device_name = "cuda", torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    device, device_name = "mps", "Apple GPU (mps)"
else:
    device, device_name = "cpu", "CPU"
print("device:", device_name)

device: CPU


## 1. Data: English subset (identical filter to the TF-IDF notebook)

In [23]:
dataset = load_dataset("deepset/prompt-injections")
DetectorFactory.seed = 0   # makes langdetect reproducible (same as the TF-IDF notebook)

def is_german(text, min_prob=0.9):
    """German = langdetect's top language is German with probability >= 0.9."""
    try:
        top = detect_langs(text)[0]
    except Exception:          # e.g. text without letters
        return False
    return top.lang == "de" and top.prob >= min_prob

def english_subset(split):
    texts, labels = list(dataset[split]["text"]), list(dataset[split]["label"])
    german = [is_german(t) for t in texts]
    kept_t = [t for t, g in zip(texts, german) if not g]
    kept_l = [l for l, g in zip(labels, german) if not g]
    print(f"{split}: {len(texts)} -> {len(kept_t)} examples ({sum(german)} removed as German); "
          f"injection share {np.mean(labels):.2f} -> {np.mean(kept_l):.2f}")
    return kept_t, kept_l

train_texts, train_labels = english_subset("train")
test_texts,  test_labels  = english_subset("test")

train: 546 -> 338 examples (208 removed as German); injection share 0.37 -> 0.41
test: 116 -> 71 examples (45 removed as German); injection share 0.52 -> 0.59


## 2. Tokeniser and prompt length
DistilBERT can read at most 512 tokens, and the training cost grows with the padded length. Check how many prompts would be cut at `MAX_LEN`. If many are, think about whether an injection hidden late in a long prompt would be lost (this is relevant for the indirect-injection discussion).

In [24]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(tokenizer(t, truncation=False)["input_ids"]) for t in train_texts])
print(pd.Series(lengths).describe(percentiles=[.5, .9, .95, .99]).round(0))
print(f"share of prompts longer than MAX_LEN={MAX_LEN}: {np.mean(lengths > MAX_LEN):.1%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (985 > 512). Running this sequence through the model will result in indexing errors


count    338.0
mean      29.0
std       64.0
min        5.0
50%       16.0
90%       51.0
95%       66.0
99%      185.0
max      985.0
dtype: float64
share of prompts longer than MAX_LEN=128: 2.4%


## 3. Functions
Written as a plain PyTorch loop (instead of the `Trainer` class) so every step is visible. We need be able to explain each of these at the exam.

In [29]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

def free_memory():
    if device == "cuda": torch.cuda.empty_cache()

def encode(texts):
    """Text -> token ids + attention mask (padded/truncated to MAX_LEN)."""
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN,
                     padding="max_length", return_tensors="pt")

def train_model(texts, labels, seed=SEED):
    """Fine-tune a fresh pretrained model on (texts, labels) and return it."""
    set_seed(seed)
    labels = np.asarray(labels)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    enc = encode(texts)
    data = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    gen = torch.Generator(); gen.manual_seed(seed)
    loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True, generator=gen)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    # class weights, the counterpart of class_weight="balanced" in the TF-IDF model
    counts = np.bincount(labels, minlength=2)
    weights = torch.tensor(len(labels) / (2 * counts), dtype=torch.float, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for ids, mask, y in loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item()
        print(f"  epoch {epoch + 1}/{EPOCHS}  mean loss {running / len(loader):.4f}")
    return model

def predict_proba(model, texts, batch_size=32):
    """Probability that each text is an injection (class 1)."""
    model.eval()
    enc = encode(texts)
    out = []
    with torch.no_grad():
        for i in range(0, len(enc["input_ids"]), batch_size):
            ids = enc["input_ids"][i:i + batch_size].to(device)
            mask = enc["attention_mask"][i:i + batch_size].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            out.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def time_single(model, texts, warmup=5):
    """Latency in ms per prompt at batch size 1 (tokenisation + forward pass)."""
    model.eval()
    texts = list(texts)
    times = []
    with torch.no_grad():
        for i, t in enumerate(texts[:warmup] + texts):   # the first few runs are warm-up
            start = time.perf_counter()
            enc = tokenizer(t, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
            model(**enc)
            sync()
            if i >= warmup:
                times.append((time.perf_counter() - start) * 1000)
    return np.array(times)

In [30]:
def metrics(y_true, scores, threshold=0.5):
    """Same definitions as in the TF-IDF notebook (injection = positive class)."""
    y = np.asarray(y_true)
    pred = (np.asarray(scores) >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"accuracy": (tp + tn) / len(y), "precision": precision, "recall": recall,
            "f1": f1, "specificity": tn / (tn + fp) if tn + fp else 0.0}

## 4. Cross-validation on the `train` split
Each fold trains a fresh model from the pretrained weights with the settings above, and the settings are not tuned on the validation fold. The out-of-fold scores are averaged over the repeats and saved as `results/<method>__val.csv`. The shared evaluation script uses them to choose a threshold at a fixed false-positive rate.

In [31]:
Path("results").mkdir(exist_ok=True)

y_train = np.asarray(train_labels)
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)

oof = np.zeros((N_REPEATS, len(train_texts)))   # out-of-fold scores, one row per repeat
rows = []
for i, (tr, va) in enumerate(cv.split(train_texts, y_train)):
    print(f"fold {i + 1}/{N_SPLITS * N_REPEATS}")
    model = train_model([train_texts[j] for j in tr], y_train[tr], seed=SEED + i)
    scores = predict_proba(model, [train_texts[j] for j in va])
    oof[i // N_SPLITS, va] = scores
    rows.append(metrics(y_train[va], scores))
    print(f"  F1 {rows[-1]['f1']:.3f}  recall {rows[-1]['recall']:.3f}  specificity {rows[-1]['specificity']:.3f}")
    del model
    free_memory()

cv_df = pd.DataFrame(rows)
print(f"\n--- {METHOD_NAME}: {len(cv_df)} folds ---")
for m in ["accuracy", "precision", "recall", "f1", "specificity"]:
    print(f"{m:22s} {cv_df[m].mean():.3f} +/- {cv_df[m].std():.3f}")
print(f"{'false_positive_rate':22s} {1 - cv_df['specificity'].mean():.3f}")

pd.DataFrame({"id": range(len(train_texts)), "label": y_train,
              "score": oof.mean(axis=0), "latency_ms": np.nan}
             ).to_csv(f"results/{METHOD_NAME}__val.csv", index=False)

fold 1/10


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3  mean loss 0.6642
  epoch 2/3  mean loss 0.4878
  epoch 3/3  mean loss 0.3477
  F1 0.949  recall 1.000  specificity 0.925
fold 2/10


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (21 outputs left out)

  epoch 1/3  mean loss 0.6447
  epoch 2/3  mean loss 0.4252
  epoch 3/3  mean loss 0.2676
  F1 0.863  recall 0.786  specificity 0.974
fold 10/10


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3  mean loss 0.6754
  epoch 2/3  mean loss 0.5711
  epoch 3/3  mean loss 0.4709
  F1 0.800  recall 1.000  specificity 0.641

--- distilbert: 10 folds ---
accuracy               0.929 +/- 0.054
precision              0.893 +/- 0.086
recall                 0.957 +/- 0.067
f1                     0.920 +/- 0.052
specificity            0.909 +/- 0.098
false_positive_rate    0.091


## 5. One final evaluation on the official `test` split
Run this once, after you have fixed the settings. Do not go back and change them because of this result.

In [32]:
final_model = train_model(train_texts, train_labels, seed=SEED)
test_scores = predict_proba(final_model, test_texts)
test_pred = (test_scores >= 0.5).astype(int)

print(classification_report(test_labels, test_pred, target_names=["benign", "injection"]))
print(confusion_matrix(test_labels, test_pred))

lat = time_single(final_model, test_texts)
print(f"\nlatency per prompt (batch size 1): median {np.median(lat):.1f} ms, "
      f"mean {lat.mean():.1f} ms, on {device_name}")

pd.DataFrame({"id": range(len(test_texts)), "label": test_labels,
              "score": test_scores, "latency_ms": lat}
             ).to_csv(f"results/{METHOD_NAME}__test.csv", index=False)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3  mean loss 0.6383
  epoch 2/3  mean loss 0.3938
  epoch 3/3  mean loss 0.1970
              precision    recall  f1-score   support

      benign       0.88      1.00      0.94        29
   injection       1.00      0.90      0.95        42

    accuracy                           0.94        71
   macro avg       0.94      0.95      0.94        71
weighted avg       0.95      0.94      0.94        71

[[29  0]
 [ 4 38]]

latency per prompt (batch size 1): median 13.8 ms, mean 16.0 ms, on CPU


## 6. Hand-written test set and second dataset
Evaluate the same `final_model` on the hand-written CSV (at least 50 per class, written before looking at errors) and on the second public dataset, once they exist. Use the same save format, with a different split name.

In [ ]:
# hw = pd.read_csv("data/handwritten/handwritten_v1.csv")
# hw_scores = predict_proba(final_model, list(hw["text"]))
# print(classification_report(hw["label"], (hw_scores >= 0.5).astype(int), target_names=["benign", "injection"]))
# print(confusion_matrix(hw["label"], (hw_scores >= 0.5).astype(int)))
# pd.DataFrame({"id": hw["id"], "label": hw["label"], "score": hw_scores,
#               "latency_ms": np.nan}).to_csv(f"results/{METHOD_NAME}__handwritten.csv", index=False)

## Things you should be able to explain (supervisor: "you will be asked at the exam")
- What a **pretrained** model is, and what **fine-tuning** changes (all weights, plus a new classification layer on top of the `[CLS]` representation).
- What the **tokeniser** does (WordPiece sub-words) and why prompts are padded and truncated to `MAX_LEN`.
- How the logits become a **score**: softmax gives P(injection), and 0.5 is only a default threshold.
- What **AdamW**, the **learning rate schedule** (warm-up, then linear decay) and **gradient clipping** are for.
- Why the **class weights** are used, and how they relate to `class_weight="balanced"` in the TF-IDF model.
- Why every fold starts from the pretrained weights, and why settings are not tuned on the validation fold or the test split.
- Why the German examples were removed, how that was done (`langdetect`, probability >= 0.9), and what it means for the conclusions (English only).

Write the final values of `MODEL_NAME, MAX_LEN, BATCH_SIZE, EPOCHS, LR, SEED` in the methodology chapter and in `docs/decisions.md`.